# YOLO26s-cls · Colab 实验

将晶圆缺陷图识别为 38 类模式，比较 E0 无增强、E1 几何增强与 E2 类别均衡采样。

**准备**：选择 GPU 运行时，将 `wafer38_v1.zip` 上传至 Drive 的 `MyDrive/WaferMap/`。

**流程**：环境与数据（1–6）→ 短试跑（7）→ 训练（8–9）→ 验证与冻结（10）→ 测试与分析（11–14）。训练、冻结和测试开关默认关闭，确认当前步骤后再开启。

数据来自 [WaferMap](https://github.com/Junliangwangdhu/WaferMap)：原始 38,015 张，修正噪声并去重后 36,991 张；训练 / 验证 / 测试为 25,893 / 5,549 / 5,549。`3→2` 为预处理假设，完全去重不排除近似重复或批次关联。

## 1. 选择 GPU 并安装依赖
Colab 菜单：**运行时 → 更改运行时类型 → GPU**。预期为 Tesla T4；其他 GPU 也可试跑，但三组应在同一型号上比较耗时。

本单元使用 Colab 自带 PyTorch，只固定 Ultralytics；若此前导入过不同版本的 Ultralytics，安装后重启会话并从头运行。

In [ ]:
import importlib.util
import os, sys, subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.189", "scikit-learn", "pandas", "matplotlib"])
    subprocess.run(["nvidia-smi"], check=True)

import torch, ultralytics, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
assert ultralytics.__version__ == "8.4.189", "请重启会话后从头运行"
assert torch.cuda.is_available(), "未获得 GPU：请更改运行时类型后重试"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__, "| ultralytics:", ultralytics.__version__)

## 2. 挂载 Drive 与设置路径
第一次运行按 Colab 提示授权挂载。`RUN_TAG` 是一次完整三组实验的名称；修改参数或重新开始实验时换一个名称。

若 T4 的 batch=32 显存不足，改为 16 并重新进行全部三组试跑。恢复时保留原参数与 RUN_TAG。

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/wafer_work")
    DRIVE_ROOT = Path("/content/drive/MyDrive/WaferMap")
else:
    # 供本地验证；实际 Colab 不使用本机目录。
    PROJECT = Path.cwd()
    while not (PROJECT / "data/processed/wafer38_v1.zip").exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent
    WORK = PROJECT / "yolo26s_cls/runs/notebook_local"
    DRIVE_ROOT = WORK / "backup"
WORK.mkdir(parents=True, exist_ok=True)
RUN_TAG = "wafer38_v1_seed42"
BATCH = 32
WORKERS = 2 if IN_COLAB else 0
EPOCHS = 100
DEVICE = 0
ARCHIVE = DRIVE_ROOT / "wafer38_v1.zip" if IN_COLAB else PROJECT / "data/processed/wafer38_v1.zip"
os.environ["YOLO_CONFIG_DIR"] = str(WORK / "ultralytics_config")
os.chdir(WORK)
print("数据 ZIP:", ARCHIVE)
print("持久备份:", DRIVE_ROOT / "runs" / RUN_TAG)

## 3. 加载共用代码

直接运行下方折叠的代码单元。它写入并导入辅助模块，包含训练、评估与恢复逻辑。

所有阶段使用完整晶圆的最近邻缩放（224×224）和 ImageNet 归一化。恢复已有实验时继续使用原 notebook 快照。

In [ ]:
# @title 生成共用实验代码（直接运行）
import importlib
MODULE_PATH = WORK / "wafer_colab.py"
MODULE_PATH.write_text('"""Shared Colab/local helpers for the three wafer classification experiments."""\nfrom copy import copy\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport time\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import DataLoader, WeightedRandomSampler\nimport torchvision\nfrom torchvision import transforms as T\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\nimport ultralytics\nfrom ultralytics import YOLO\nfrom ultralytics.data.build import InfiniteDataLoader, seed_worker\nfrom ultralytics.models.yolo.classify import ClassificationTrainer, ClassificationValidator\nfrom ultralytics.utils.metrics import ClassifyMetrics\nfrom ultralytics.utils.torch_utils import EarlyStopping\n\nDATA_SHA = \'bc1ecce02371a39d68533008aae57ef5c7d21954b795d22e917e5cb40851bd62\'\nEXPERIMENTS = {\'E0\': \'baseline\', \'E1\': \'geometry\', \'E2\': \'balanced\'}\nMEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef dump(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + \'\\n\', encoding=\'utf-8\')\n\n\ndef atomic_copy(source, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temp = target.with_name(target.name + \'.tmp\')\n    shutil.copy2(source, temp)\n    temp.replace(target)\n\n\ndef unpack(archive, destination):\n    archive, destination = Path(archive), Path(destination)\n    if sha(archive) != DATA_SHA:\n        raise ValueError(\'数据 ZIP SHA-256 不匹配，请使用本项目的 wafer38_v1.zip\')\n    root = destination / \'wafer38_v1\'\n    if not root.exists():\n        with zipfile.ZipFile(archive) as z:\n            for info in z.infolist():\n                path = destination / info.filename\n                if not path.resolve().is_relative_to(destination.resolve()) or not info.filename.startswith(\'wafer38_v1/\'):\n                    raise ValueError(\'ZIP contains an unexpected path\')\n            if z.testzip() is not None:\n                raise ValueError(\'ZIP CRC check failed\')\n            z.extractall(destination)\n    return root\n\n\ndef read_metadata(root):\n    root = Path(root)\n    classes = json.loads((root / \'metadata/classes.json\').read_text(encoding=\'utf-8\'))\n    with (root / \'metadata/manifest.csv\').open(encoding=\'utf-8\', newline=\'\') as f:\n        rows = list(csv.DictReader(f))\n    if [c[\'folder\'] for c in classes] != [f\'class_{i:02d}\' for i in range(38)]:\n        raise ValueError(\'类别顺序必须为 class_00 ... class_37\')\n    return classes, rows\n\n\ndef audit(root):\n    root = Path(root)\n    classes, rows = read_metadata(root)\n    summary = json.loads((root / \'metadata/summary.json\').read_text(encoding=\'utf-8\'))\n    if len(rows) != summary[\'unique_samples\']:\n        raise ValueError(\'Manifest count mismatch\')\n    hashes, counts = {}, {}\n    for row in rows:\n        relative = Path(row[\'path\'])\n        target = root / relative\n        class_id, split = int(row[\'class_id\']), row[\'split\']\n        if split not in (\'train\', \'val\', \'test\') or not 0 <= class_id < 38:\n            raise ValueError(\'Invalid split or class\')\n        if not target.resolve().is_relative_to(root.resolve()) or relative.parts[:2] != (split, classes[class_id][\'folder\']):\n            raise ValueError(\'Invalid manifest path\')\n        if sha(target) != row[\'png_sha256\'] or row[\'matrix_sha256\'] in hashes:\n            raise ValueError(\'PNG changed or duplicate matrix found\')\n        hashes[row[\'matrix_sha256\']] = split\n        counts[split] = counts.get(split, 0) + 1\n    for split, count in counts.items():\n        if count != summary[\'split_counts\'][split]:\n            raise ValueError(\'Split count mismatch\')\n        if len(list((root / split).glob(\'*/*.png\'))) != count:\n            raise ValueError(\'Unexpected images in split\')\n        if {int(r[\'class_id\']) for r in rows if r[\'split\'] == split} != set(range(38)):\n            raise ValueError(\'Missing class\')\n    return summary\n\n\ndef make_smoke(root, output, train_per_class=30, val_per_class=10):\n    root, output = Path(root), Path(output)\n    if output.exists():\n        read_metadata(output)\n        return output\n    classes, rows = read_metadata(root)\n    selected = []\n    for split, limit in [(\'train\', train_per_class), (\'val\', val_per_class)]:\n        for c in classes:\n            candidates = [r for r in rows if r[\'split\'] == split and int(r[\'class_id\']) == c[\'class_id\']]\n            for row in candidates[:limit]:\n                target = output / row[\'path\']\n                target.parent.mkdir(parents=True, exist_ok=True)\n                shutil.copy2(root / row[\'path\'], target)\n                selected.append(row)\n    (output / \'metadata\').mkdir()\n    shutil.copy2(root / \'metadata/classes.json\', output / \'metadata/classes.json\')\n    with (output / \'metadata/manifest.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(selected)\n    return output\n\n\ndef transform(geometry=False):\n    operations = []\n    if geometry:\n        operations += [T.RandomChoice([T.RandomRotation((a, a), interpolation=T.InterpolationMode.NEAREST)\n                                       for a in (0, 90, 180, 270)]), T.RandomHorizontalFlip(0.5)]\n    return T.Compose(operations + [T.Resize((224, 224), interpolation=T.InterpolationMode.NEAREST),\n                                   T.ToTensor(), T.Normalize(MEAN, STD)])\n\n\nclass MacroMetrics(ClassifyMetrics):\n    def __init__(self):\n        super().__init__()\n        self.macro_f1 = 0.0\n\n    @property\n    def keys(self):\n        return super().keys + [\'metrics/macro_f1\']\n\n    @property\n    def fitness(self):\n        return self.macro_f1\n\n    @property\n    def results_dict(self):\n        return dict(zip(self.keys + [\'fitness\'], [self.top1, self.top5, self.macro_f1, self.fitness]))\n\n\nclass WaferValidator(ClassificationValidator):\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.metrics = MacroMetrics()\n\n    def build_dataset(self, img_path):\n        dataset = super().build_dataset(img_path)\n        dataset.torch_transforms = transform()\n        return dataset\n\n    def get_stats(self):\n        self.metrics.process(self.targets, self.pred)\n        true = torch.cat(self.targets).numpy()\n        pred = torch.cat(self.pred)[:, 0].numpy()\n        self.metrics.macro_f1 = float(f1_score(true, pred, labels=list(range(38)), average=\'macro\', zero_division=0))\n        return self.metrics.results_dict\n\n    def print_results(self):\n        print(f\'Validation: Top1={self.metrics.top1:.4f} Macro-F1={self.metrics.macro_f1:.4f}\')\n\n\nclass StrictEarlyStopping(EarlyStopping):\n    def __call__(self, epoch, fitness):\n        if fitness == self.best_fitness:\n            delta = epoch - self.best_epoch\n            self.possible_stop = delta >= self.patience - 1\n            return delta >= self.patience\n        return super().__call__(epoch, fitness)\n\n\nclass WaferTrainer(ClassificationTrainer):\n    def __init__(self, *args, experiment=\'E0\', **kwargs):\n        self.experiment = experiment\n        super().__init__(*args, **kwargs)\n\n    def build_dataset(self, img_path, mode=\'train\', batch=None):\n        dataset = super().build_dataset(img_path, mode, batch)\n        dataset.torch_transforms = transform(mode == \'train\' and self.experiment == \'E1\')\n        return dataset\n\n    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode=\'train\'):\n        if self.experiment != \'E2\' or mode != \'train\':\n            return super().get_dataloader(dataset_path, batch_size, rank, mode)\n        if rank not in (-1, 0):\n            raise ValueError(\'This notebook supports one GPU\')\n        dataset = self.build_dataset(dataset_path, mode)\n        labels = np.array([s[1] for s in dataset.samples])\n        counts = np.bincount(labels, minlength=38)\n        if (counts == 0).any():\n            raise ValueError(\'Training split missing classes\')\n        sampler = WeightedRandomSampler(torch.as_tensor(1.0 / counts[labels]), len(labels), replacement=True)\n        return InfiniteDataLoader(dataset, batch_size=batch_size, sampler=sampler,\n                                  num_workers=self.args.workers, pin_memory=self.device.type == \'cuda\',\n                                  worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(42))\n\n    def get_validator(self):\n        validator = WaferValidator(self.test_loader, self.save_dir, args=copy(self.args), _callbacks=self.callbacks)\n        return validator\n\n    def _setup_train(self):\n        super()._setup_train()\n        stopper = StrictEarlyStopping(patience=self.args.patience)\n        stopper.__dict__.update(self.stopper.__dict__)\n        self.stopper = stopper\n\n    def final_eval(self):\n        # Ultralytics overwrites best.pt on ties; retain the earliest strict Macro-F1 maximum.\n        selected = self.wdir / \'best_macro.pt\'\n        if selected.exists():\n            shutil.copy2(selected, self.best)\n        return super().final_eval()\n\n\ndef environment():\n    return dict(torch=torch.__version__, torchvision=torchvision.__version__, ultralytics=ultralytics.__version__,\n                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\')\n\n\ndef run_experiment(root, output, experiment, *, backup=None, resume=False, epochs=100, batch=32,\n                   workers=2, device=0, stop_after=None, model=\'yolo26s-cls.pt\'):\n    """Train one experiment; backup contains a pre-strip resume.pt after every epoch."""\n    if experiment not in EXPERIMENTS or ultralytics.__version__ != \'8.4.189\':\n        raise ValueError(\'Require E0/E1/E2 and ultralytics==8.4.189\')\n    if model not in (\'yolo26n-cls.pt\', \'yolo26s-cls.pt\'):\n        raise ValueError(\'Model must be yolo26n-cls.pt or yolo26s-cls.pt\')\n    root, output = Path(root).resolve(), Path(output).resolve()\n    classes, _ = read_metadata(root)\n    identity = dict(experiment=experiment, model=model, manifest_sha256=sha(root / \'metadata/manifest.csv\'),\n                    classes_sha256=sha(root / \'metadata/classes.json\'), code_sha256=sha(__file__),\n                    environment=environment(), epochs=epochs, batch=batch)\n    backup = Path(backup).resolve() if backup is not None else None\n    if backup == output:\n        raise ValueError(\'Backup directory must differ from working directory\')\n    if resume:\n        if not (output / \'run.json\').exists() and backup is not None:\n            shutil.copytree(backup, output)\n        previous = json.loads((output / \'run.json\').read_text(encoding=\'utf-8\'))\n        if previous != identity:\n            raise ValueError(\'恢复要求实验、数据、代码、软件环境、轮数及 batch 与原记录一致\')\n        checkpoint = output / \'weights/resume.pt\'\n        if not checkpoint.exists():\n            raise FileNotFoundError(\'没有可恢复检查点；请先完成一轮训练\')\n        checkpoint_state = torch.load(checkpoint, map_location=\'cpu\', weights_only=False)\n        if checkpoint_state[\'epoch\'] + 1 >= epochs or checkpoint_state.get(\'optimizer\') is None:\n            raise ValueError(\'训练已完成或检查点缺少优化器，不能继续恢复\')\n        del checkpoint_state\n    else:\n        if output.exists() or (backup is not None and backup.exists()):\n            raise FileExistsError(\'实验目录已存在；使用 resume=True 或更换 RUN_TAG\')\n        output.mkdir(parents=True)\n        dump(output / \'run.json\', identity)\n        for name in (\'classes.json\', \'manifest.csv\'):\n            shutil.copy2(root / \'metadata\' / name, output / name)\n        shutil.copy2(__file__, output / \'wafer_colab.py\')\n        checkpoint = model\n    selection_path = output / \'selection.json\'\n    selection = json.loads(selection_path.read_text()) if selection_path.exists() else dict(macro_f1=-1.0, epoch=0)\n    clock = {}\n\n    def epoch_start(trainer):\n        clock[\'start\'] = time.perf_counter()\n        if trainer.device.type == \'cuda\':\n            torch.cuda.reset_peak_memory_stats(trainer.device)\n\n    def saved(trainer):\n        metric = float(trainer.metrics[\'metrics/macro_f1\'])\n        if not math.isfinite(metric):\n            raise ValueError(\'Validation Macro-F1 is not finite\')\n        if metric > selection[\'macro_f1\']:\n            selection.update(macro_f1=metric, epoch=trainer.epoch + 1)\n            atomic_copy(trainer.last, output / \'weights/best_macro.pt\')\n            dump(selection_path, selection)\n        atomic_copy(trainer.last, output / \'weights/resume.pt\')\n        profile = dict(epoch=trainer.epoch + 1, seconds=time.perf_counter()-clock[\'start\'],\n                       batch=trainer.batch_size, macro_f1=metric,\n                       peak_gpu_mb=torch.cuda.max_memory_allocated(trainer.device)/2**20 if trainer.device.type==\'cuda\' else 0)\n        if trainer.batch_size != batch:\n            raise RuntimeError(\'框架自动更改了 batch；请统一三组 batch 后重新试跑\')\n        with (output / \'profile.jsonl\').open(\'a\', encoding=\'utf-8\') as f:\n            f.write(json.dumps(profile) + \'\\n\')\n        if backup is not None:\n            for p in output.rglob(\'*\'):\n                if p.is_file() and p.suffix in (\'.json\', \'.jsonl\', \'.yaml\', \'.csv\', \'.pt\', \'.py\'):\n                    atomic_copy(p, backup / p.relative_to(output))\n        if stop_after is not None and trainer.epoch + 1 >= stop_after:\n            trainer.stop = True\n\n    overrides = dict(model=str(checkpoint), data=str(root), project=str(output.parent), name=output.name,\n                     save_dir=str(output), exist_ok=True, epochs=epochs, batch=batch, imgsz=224,\n                     optimizer=\'AdamW\', lr0=0.001, lrf=0.01, cos_lr=True, warmup_epochs=3,\n                     weight_decay=0.0005, patience=15, amp=True, dropout=0.0, seed=42, deterministic=True,\n                     workers=workers, device=device, save=True, plots=False, val=True, cache=False,\n                     auto_augment=None, erasing=0.0, fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0,\n                     hsv_v=0.0, close_mosaic=0, resume=str(checkpoint) if resume else False)\n    trainer = WaferTrainer(overrides=overrides, experiment=experiment)\n    trainer.add_callback(\'on_train_epoch_start\', epoch_start)\n    trainer.add_callback(\'on_model_save\', saved)\n    trainer.train()\n    if stop_after is None or trainer.epoch + 1 >= epochs:\n        dump(output / \'completed.json\', dict(epochs_completed=trainer.epoch + 1,\n                                             best=selection, state=\'finished\'))\n    if backup is not None:\n        for p in output.rglob(\'*\'):\n            if p.is_file() and p.suffix in (\'.json\', \'.jsonl\', \'.yaml\', \'.csv\', \'.pt\', \'.py\'):\n                atomic_copy(p, backup / p.relative_to(output))\n    return output\n\n\ndef load_model(weights, root, device=0):\n    classes, _ = read_metadata(root)\n    model = YOLO(str(weights)).model.to(device).float().eval()\n    if [model.names[i] for i in range(38)] != [c[\'folder\'] for c in classes]:\n        raise ValueError(\'Model class mapping differs from dataset\')\n    return model\n\n\n@torch.inference_mode()\ndef probabilities(model, inputs):\n    output = model(inputs)\n    output = output[0] if isinstance(output, (tuple, list)) else output\n    if output.ndim != 2 or output.shape[1] != 38 or not torch.isfinite(output).all():\n        raise ValueError(\'Invalid model output\')\n    if not torch.allclose(output.sum(1), torch.ones(len(output), device=output.device), atol=1e-3):\n        raise ValueError(\'Expected classification probabilities\')\n    return output\n\n\ndef evaluate(weights, root, split=\'val\', *, device=0, batch=32, output=None):\n    if split not in (\'val\', \'test\'):\n        raise ValueError(\'Evaluation split must be val or test\')\n    root = Path(root)\n    classes, rows = read_metadata(root)\n    dataset = torchvision.datasets.ImageFolder(root / split, transform=transform())\n    if dataset.classes != [c[\'folder\'] for c in classes]:\n        raise ValueError(\'ImageFolder class mapping differs\')\n    model = load_model(weights, root, device)\n    loader = DataLoader(dataset, batch_size=batch, shuffle=False, num_workers=0)\n    true, probs = [], []\n    for images, labels in loader:\n        with torch.inference_mode():\n            p = probabilities(model, images.to(device))\n        probs.extend(p.cpu().numpy())\n        true.extend(labels.tolist())\n    p = np.asarray(probs)\n    true, pred = np.asarray(true), p.argmax(1)\n    names = [c[\'name\'] for c in classes]\n    report = classification_report(true, pred, labels=list(range(38)), target_names=names, zero_division=0, output_dict=True)\n    pattern_sizes = np.array([sum(c[\'label\']) for c in classes])\n    groups = {}\n    for name, mask in [(\'normal\', pattern_sizes[true]==0), (\'single\', pattern_sizes[true]==1), (\'mixed\', pattern_sizes[true]>=2)]:\n        groups[name] = dict(samples=int(mask.sum()), accuracy=float((true[mask]==pred[mask]).mean()))\n    metrics = dict(split=split, weights_sha256=sha(weights), manifest_sha256=sha(root/\'metadata/manifest.csv\'),\n                   samples=len(true), accuracy=float(accuracy_score(true, pred)),\n                   macro_f1=float(f1_score(true, pred, labels=list(range(38)), average=\'macro\', zero_division=0)),\n                   groups=groups, classification_report=report)\n    if output is not None:\n        output = Path(output)\n        output.mkdir(parents=True, exist_ok=True)\n        dump(output / f\'{split}_metrics.json\', metrics)\n        np.savez_compressed(output / f\'{split}_predictions.npz\', true=true, probabilities=p)\n        cm = confusion_matrix(true, pred, labels=list(range(38)))\n        np.savetxt(output / f\'{split}_confusion.csv\', cm, fmt=\'%d\', delimiter=\',\')\n        by_path = {r[\'path\']: r for r in rows}\n        with (output / f\'{split}_predictions.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n            writer = csv.writer(f)\n            writer.writerow([\'original_index\', \'path\', \'true_id\', \'pred_id\', \'confidence\', \'correct\'])\n            for (path, _), t, pred_id, confidence in zip(dataset.samples, true, pred, p.max(1)):\n                relative = Path(path).relative_to(root).as_posix()\n                writer.writerow([by_path[relative][\'original_index\'], relative, int(t), int(pred_id), float(confidence), bool(t==pred_id)])\n    return metrics\n\n\ndef predict_image(model, image):\n    if not isinstance(image, Image.Image):\n        image = Image.open(image)\n    pixels = np.asarray(image.convert(\'RGB\'))\n    if pixels.shape != (52, 52, 3) or not np.isin(pixels, [0, 128, 255]).all() or not (pixels==pixels[..., :1]).all():\n        raise ValueError(\'演示输入要求本项目 52x52 RGB PNG 编码；不是普通晶圆照片\')\n    with torch.inference_mode():\n        return probabilities(model, transform()(image.convert(\'RGB\')).unsqueeze(0).to(next(model.parameters()).device))[0].cpu().numpy()\n\n\ndef benchmark(weights, root, *, device=0):\n    model = load_model(weights, root, device)\n    _, rows = read_metadata(root)\n    row = next(r for r in rows if r[\'split\']==\'val\')\n    image = Image.open(Path(root)/row[\'path\']).convert(\'RGB\')\n    preprocess = transform()\n    def iteration():\n        with torch.inference_mode():\n            probabilities(model, preprocess(image).unsqueeze(0).to(device))\n        if next(model.parameters()).is_cuda:\n            torch.cuda.synchronize()\n    for _ in range(20):\n        iteration()\n    timings = []\n    for _ in range(200):\n        start = time.perf_counter()\n        iteration()\n        timings.append((time.perf_counter()-start)*1000)\n    return dict(batch=1, warmup=20, repeats=200, mean_ms=float(np.mean(timings)),\n                median_ms=float(np.median(timings)), p95_ms=float(np.percentile(timings,95)),\n                includes=\'PIL transform + host-to-device + fp32 model + synchronization; excludes disk read\',\n                environment=environment())\n', encoding="utf-8")
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import wafer_colab as wc
importlib.reload(wc)
print("共用代码已就绪:", MODULE_PATH)

## 4. 复制、解压并完整校验数据
复制压缩包到 `/content/` 后训练，减少 Drive 小文件读取开销。核验固定 ZIP 哈希、每张 PNG 哈希、类别顺序和集合之间的重复泄漏；此步骤可能需要数分钟。

若提示 ZIP 哈希不符，使用本项目导出的 `wafer38_v1.zip`，不要改动或自行重压缩。

In [ ]:
import shutil, json
local_zip = WORK / "wafer38_v1.zip"
if not local_zip.exists():
    shutil.copy2(ARCHIVE, local_zip)
DATA = wc.unpack(local_zip, WORK / "datasets")
summary = wc.audit(DATA)
classes, manifest = wc.read_metadata(DATA)
display(pd.DataFrame([{"split": s, "images": n} for s, n in summary["split_counts"].items()]))
print("数据校验通过，唯一晶圆数:", summary["unique_samples"])
print("本实验参数:", {"batch": BATCH, "epochs": EPOCHS, "seed": 42})

## 5. 查看类别分布与样本
去重后 36 类各有 1,000 张，Random 为 850 张、Near_Full 为 141 张。确认晶圆图四周完整，图中无标题或坐标轴。

类别 ID 采用标签向量字典序，与原论文 C1～C38 编号不等价；汇报使用这里保存的名称和类别映射。

In [ ]:
counts = pd.DataFrame(summary["classes"])
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(counts["folder"], counts[["train", "val", "test"]].sum(axis=1), color="#336699")
ax.set(ylabel="Unique wafers", title="Wafer38: class counts after cleaning and deduplication")
ax.tick_params(axis="x", labelrotation=90)
fig.tight_layout(); plt.show()
display(counts.loc[counts["folder"].isin(["class_01", "class_03"]), ["folder", "name", "train", "val", "test"]])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, class_id in zip(axes.flat, [0, 1, 2, 3, 4, 6, 10, 37]):
    row = next(r for r in manifest if r["split"] == "train" and int(r["class_id"]) == class_id)
    ax.imshow(Image.open(DATA / row["path"]), interpolation="nearest")
    ax.set_title(f"{classes[class_id]['folder']}\n{classes[class_id]['name']}", fontsize=10)
    ax.axis("off")
fig.tight_layout(); plt.show()

## 6. 检查输入变换
整图四角应保留，最近邻缩放不产生新灰度；E1 只增加 90°倍数旋转和水平翻转，保持模式标签含义。

In [ ]:
row = next(r for r in manifest if r["split"] == "train")
image = Image.open(DATA / row["path"]).convert("RGB")
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, geometry, title in zip(axes, [False, False, True], ["Original 52x52", "Full image 224x224", "E1 geometry example"]):
    if title.startswith("Original"):
        shown = np.asarray(image)
    else:
        tensor = wc.transform(geometry)(image)
        shown = (tensor * torch.tensor(wc.STD)[:, None, None] + torch.tensor(wc.MEAN)[:, None, None]).permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(shown, interpolation="nearest"); ax.set_title(title); ax.axis("off")
fig.tight_layout(); plt.show()

## 7. 短试跑与恢复验证

将 `RUN_SMOKE=True`：每类最多 30 张训练图、10 张验证图，三组各跑 2 轮；E0 在第 1 轮后暂停并恢复，不使用测试集。

确认 loss、Macro-F1、显存及 Drive 备份正常，再进入正式训练。恢复使用含优化器状态的 `resume.pt`；不保证中断前后逐位相同。AMP 检查可能另下载小型检测模型。

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    smoke = wc.make_smoke(DATA, WORK / "smoke_data")
    for experiment in wc.EXPERIMENTS:
        run = WORK / "smoke_runs" / RUN_TAG / experiment
        backup = DRIVE_ROOT / "smoke_runs" / RUN_TAG / experiment
        if not (run / "run.json").exists() and not (backup / "run.json").exists():
            wc.run_experiment(smoke, run, experiment, backup=backup, epochs=2, batch=BATCH,
                              workers=WORKERS, device=DEVICE, stop_after=1 if experiment == "E0" else None)
        checkpoint = run / "weights/resume.pt"
        if not checkpoint.exists() and (backup / "weights/resume.pt").exists():
            shutil.copytree(backup, run, dirs_exist_ok=True)
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
        completed = state["epoch"] + 1
        del state
        if completed < 2:
            wc.run_experiment(smoke, run, experiment, backup=backup, resume=True, epochs=2,
                              batch=BATCH, workers=WORKERS, device=DEVICE)
        display(pd.read_json(run / "profile.jsonl", lines=True))
    print("三组短试跑已完成。确认上方结果和 Drive 文件后，再开启正式训练。")
else:
    print("已跳过短试跑；首次使用应先开启。")

## 8. 正式训练：一次选择一个实验

| 实验 | 改动 | 共同设置 |
|---|---|---|
| E0 | 无增强、普通打乱 | 同一预训练权重与划分 |
| E1 | 90°倍数旋转＋水平翻转 | 其余同 E0 |
| E2 | 无增强、按类别频数倒数加权采样 | 每轮采样量等于训练集大小 |

AdamW，lr=0.001，weight decay=0.0005，3 轮预热与余弦下降，最多 100 轮；验证 Macro-F1 连续 15 轮无提升时早停。严格提升才更新 `best_macro.pt`，同分保留较早权重。

第一次：设置 `RUN_TRAINING=True`，`EXPERIMENT="E0"`。完成后分别改为 E1、E2 运行本单元。**不要同时启动多个 Notebook 使用同一 RUN_TAG。** 每轮复制权重与日志到 Drive；备份失败会报错。

恢复：重新连接 GPU，从第 1 步开始，跳过已完成短试跑，保持 RUN_TAG、EXPERIMENT、batch 和轮数不变，将 `RESUME=True`。完整训练结束后的检查点不再恢复。

In [ ]:
RUN_TRAINING = False
EXPERIMENT = "E0"  # E0 / E1 / E2
RESUME = False
if RUN_TRAINING:
    run = WORK / "runs" / RUN_TAG / EXPERIMENT
    backup = DRIVE_ROOT / "runs" / RUN_TAG / EXPERIMENT
    wc.run_experiment(DATA, run, EXPERIMENT, backup=backup, resume=RESUME, epochs=EPOCHS,
                      batch=BATCH, workers=WORKERS, device=DEVICE)
    print("已完成:", backup)
else:
    print("正式训练未启动；先完成短试跑，再设置 RUN_TRAINING=True。")

## 9. 加载正式实验与训练曲线
重新连接后，这一步也可直接使用 Drive 上的权重和记录。只加载正式实验目录，不把短试跑作为正式成绩。

In [ ]:
RUNS = {}
for experiment in wc.EXPERIMENTS:
    local = WORK / "runs" / RUN_TAG / experiment
    saved = DRIVE_ROOT / "runs" / RUN_TAG / experiment
    directory = local if (local / "weights/best_macro.pt").exists() else saved
    if (directory / "weights/best_macro.pt").exists() and (directory / "completed.json").exists():
        RUNS[experiment] = directory
print("已找到正式权重:", list(RUNS))
if RUNS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for experiment, directory in RUNS.items():
        log = pd.read_csv(directory / "results.csv").rename(columns=lambda x: x.strip())
        axes[0].plot(log["epoch"], log["train/loss"], label=experiment)
        axes[1].plot(log["epoch"], log["metrics/macro_f1"], label=experiment)
    axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross entropy")
    axes[1].set(title="Validation Macro-F1", xlabel="Epoch", ylabel="Macro-F1", ylim=(0, 1))
    for ax in axes: ax.legend(); ax.grid(alpha=0.2)
    fig.tight_layout(); plt.show()
else:
    print("完成正式训练后重跑本单元。")

## 10. 验证集比较与冻结最终方案
按验证 Macro-F1、准确率排序；仍同分时依次优先 E0、E1、E2。只有三组全部完成后才冻结，写入最终权重哈希和配置。比较使用重新加载的最佳权重，不依赖 Notebook 中残留的模型。

所有调参到这里结束。单随机种子差异不代表统计显著改善。

In [ ]:
RUN_VALIDATION = False
FREEZE_FINAL = False
REPORTS = DRIVE_ROOT / "reports" / RUN_TAG
if RUN_VALIDATION:
    if set(RUNS) != set(wc.EXPERIMENTS):
        raise RuntimeError("请先完成三组正式实验并运行第 9 步")
    comparison = []
    for experiment, directory in RUNS.items():
        result = wc.evaluate(directory / "weights/best_macro.pt", DATA, "val", device=DEVICE, batch=BATCH,
                             output=REPORTS / experiment)
        comparison.append({"experiment": experiment, "accuracy": result["accuracy"], "macro_f1": result["macro_f1"]})
    comparison = pd.DataFrame(comparison).sort_values(["macro_f1", "accuracy", "experiment"], ascending=[False, False, True])
    REPORTS.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(REPORTS / "validation_comparison.csv", index=False)
    display(comparison)
    if FREEZE_FINAL:
        frozen_path = REPORTS / "frozen.json"
        if frozen_path.exists():
            raise FileExistsError("方案已冻结；保留记录，不能用测试结果重新选模型")
        winner = comparison.iloc[0]["experiment"]
        weights = RUNS[winner] / "weights/best_macro.pt"
        wc.atomic_copy(weights, REPORTS / "final.pt")
        wc.dump(frozen_path, {"experiment": winner, "weights_sha256": wc.sha(REPORTS / "final.pt"),
                              "validation": comparison.iloc[0].to_dict(), "data_sha256": wc.DATA_SHA})
        print("已冻结:", winner, REPORTS / "final.pt")
else:
    print("正式训练完成后设置 RUN_VALIDATION=True；确认比较结果后再设置 FREEZE_FINAL=True。")

## 11. 最终测试（方案冻结后）
显式开启后评估三组最佳模型，输出准确率、Macro-F1、逐类指标、分组准确率、预测清单及混淆矩阵。测试结果用于报告，不再用于挑选或调整方案。已生成结果时读取缓存，避免重复计算。

本 Notebook 按验证集选择最终模型；即使测试集另一组分数更高，也保留冻结方案。

In [ ]:
RUN_TEST = False
if RUN_TEST:
    frozen = json.loads((REPORTS / "frozen.json").read_text(encoding="utf-8"))
    assert wc.sha(REPORTS / "final.pt") == frozen["weights_sha256"]
    assert set(RUNS) == set(wc.EXPERIMENTS)
    tests = []
    for experiment, directory in RUNS.items():
        metric_path = REPORTS / experiment / "test_metrics.json"
        weights = directory / "weights/best_macro.pt"
        if metric_path.exists():
            result = json.loads(metric_path.read_text(encoding="utf-8"))
            assert result["weights_sha256"] == wc.sha(weights), "已有结果对应不同权重"
        else:
            result = wc.evaluate(weights, DATA, "test", device=DEVICE, batch=BATCH, output=REPORTS / experiment)
        tests.append({"experiment": experiment, "accuracy": result["accuracy"], "macro_f1": result["macro_f1"],
                      "frozen_final": experiment == frozen["experiment"]})
    display(pd.DataFrame(tests))
else:
    print("测试集尚未评估。")

## 12. 混淆矩阵、逐类指标与错误案例
矩阵行是真实类别、列是预测类别；归一化图按行除以该类样本数。类别 ID 的完整含义见下方表格。优先检查少数类 Near_Full 的召回率及高置信度误判。

In [ ]:
if (REPORTS / "frozen.json").exists():
    frozen = json.loads((REPORTS / "frozen.json").read_text(encoding="utf-8"))
    experiment = frozen["experiment"]
    result_dir = REPORTS / experiment
    if (result_dir / "test_metrics.json").exists():
        metrics = json.loads((result_dir / "test_metrics.json").read_text(encoding="utf-8"))
        display(pd.DataFrame(metrics["classification_report"]).T)
        display(pd.DataFrame(metrics["groups"]).T)
        cm = np.loadtxt(result_dir / "test_confusion.csv", delimiter=",")
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for ax, values, title in zip(axes, [cm, cm / cm.sum(1, keepdims=True)], ["Confusion counts", "Recall by true class"]):
            shown = ax.imshow(values, cmap="Blues", vmin=0)
            ax.set(title=title, xlabel="Predicted class ID", ylabel="True class ID")
            ax.set_xticks(range(0, 38, 2)); ax.set_yticks(range(0, 38, 2))
            fig.colorbar(shown, ax=ax, fraction=0.046)
        fig.tight_layout(); fig.savefig(result_dir / "confusion.png", dpi=180); plt.show()
        errors = pd.read_csv(result_dir / "test_predictions.csv")
        errors = errors.loc[~errors["correct"]].sort_values("confidence", ascending=False)
        errors.to_csv(result_dir / "errors.csv", index=False)
        display(errors.head(10))
        if len(errors):
            fig, axes = plt.subplots(2, 3, figsize=(12, 7))
            for ax, (_, row) in zip(axes.flat, errors.head(6).iterrows()):
                ax.imshow(Image.open(DATA / row["path"]), interpolation="nearest")
                ax.set_title(f"True: {int(row['true_id']):02d} / Pred: {int(row['pred_id']):02d}\nscore={row['confidence']:.3f}")
            for ax in axes.flat: ax.axis("off")
            fig.tight_layout(); fig.savefig(result_dir / "error_examples.png", dpi=180); plt.show()
        else:
            print("本次测试没有错误样本，如实报告，不构造错误案例。")
        display(pd.DataFrame(classes)[["class_id", "name"]])
    else:
        print("请先完成最终测试。")
else:
    print("请先完成验证比较并冻结方案。")

## 13. 推理速度与单图演示
速度在同一设备上测量：batch=1，预热 20 次、计时 200 次，包含 CPU 图片变换、传输、FP32 前向及 CUDA 同步，不含磁盘读取。分数是模型输出，不是校准后的可靠性概率。

演示选择数据集中的原始索引，或读取同一 52×52、RGB 三通道一致、0/128/255 编码的 PNG；不能直接输入普通晶圆照片。

In [ ]:
RUN_BENCHMARK = False
DEMO_ORIGINAL_INDEX = None  # None 自动选测试集第一张；可填原始索引整数
if (REPORTS / "final.pt").exists():
    if RUN_BENCHMARK:
        speed_rows = []
        for experiment, directory in RUNS.items():
            speed = wc.benchmark(directory / "weights/best_macro.pt", DATA, device=DEVICE)
            wc.dump(REPORTS / experiment / "speed.json", speed)
            speed_rows.append({"experiment": experiment, "mean_ms": speed["mean_ms"], "p95_ms": speed["p95_ms"]})
        display(pd.DataFrame(speed_rows))
    model = wc.load_model(REPORTS / "final.pt", DATA, DEVICE)
    row = next(r for r in manifest if r["split"] == "test" and
               (DEMO_ORIGINAL_INDEX is None or int(r["original_index"]) == DEMO_ORIGINAL_INDEX))
    image = Image.open(DATA / row["path"]).convert("RGB")
    scores = wc.predict_image(model, image)
    top3 = scores.argsort()[::-1][:3]
    plt.figure(figsize=(4, 4)); plt.imshow(image, interpolation="nearest"); plt.axis("off")
    plt.title(f"True: {classes[int(row['class_id'])]['name']}"); plt.show()
    display(pd.DataFrame([{"rank": rank+1, "class": classes[i]["name"], "score": float(scores[i])} for rank, i in enumerate(top3)]))
else:
    print("冻结模型后可使用本单元演示。")

## 14. 保存与复现

- Drive 的 `WaferMap/` 下的 `runs/RUN_TAG/` 保存配置、日志和权重，`reports/RUN_TAG/` 保存评估结果。
- `best_macro.pt` 用于评估；`resume.pt` 包含优化器状态，用于恢复。
- 保存带输出的实验副本；提交到 Git 的 notebook 保持输出清空。
- 新实验更换 `RUN_TAG`；已有目录选择恢复，不直接删除。备份失败时先检查 Drive 挂载与空间。

报告注明划分、随机种子、软件版本和设备；不同设备耗时不能直接归因于模型大小。